### Note: Scraping code change from 1st Feb 2021, because the site was updated

In [25]:
import requests
from bs4 import BeautifulSoup
import re
import ast
import json
import pandas as pd
from pandas.io.json import json_normalize
from datetime import date

# Setup

In [26]:
# Request URL
page = requests.get("https://fastandfresh.in/collections/all")

In [27]:
# Fetch webpage
soup = BeautifulSoup(page.content,"html.parser")

In [28]:
# find number of pages on main site
page_links = soup.findAll("div", {"class": "Pagination__Nav"})
#page_nums = [int(x.string) for x in page_links[0].findAll("span", {"class": "Pagination__NavItem Link Link--primary"})]
page_nums = [x for x in range(1,11)]

pages = ['https://fastandfresh.in/collections/all?page='+str(x) for x in range(1,max(page_nums)+1)]

In [29]:
pages

['https://fastandfresh.in/collections/all?page=1',
 'https://fastandfresh.in/collections/all?page=2',
 'https://fastandfresh.in/collections/all?page=3',
 'https://fastandfresh.in/collections/all?page=4',
 'https://fastandfresh.in/collections/all?page=5',
 'https://fastandfresh.in/collections/all?page=6',
 'https://fastandfresh.in/collections/all?page=7',
 'https://fastandfresh.in/collections/all?page=8',
 'https://fastandfresh.in/collections/all?page=9',
 'https://fastandfresh.in/collections/all?page=10']

# Scrape

In [30]:
%%time

data_list = []

# for each page on items available
for p in pages:
    print (p)
    req = requests.get(p)
    psoup = BeautifulSoup(req.content,"html.parser")
    anchors = psoup.findAll("a", {"class": "ProductItem__ImageWrapper"})
    
    # search through each item's page
    for anchor in anchors:
        link = anchor['href']
        print (link)
        cont = BeautifulSoup(requests.get("https://fastandfresh.in"+link).content,"html.parser")
        data = json.loads(cont.findAll('script', type='application/ld+json')[0].string[1:-1])
        data.update(data['offers'][0])
        data_list.append(data)
        #print (data)
    
    print ('\n')

https://fastandfresh.in/collections/all?page=1
/collections/all/products/amla
/collections/all/products/anar-kandhari
/collections/all/products/apple-green-imported-price-per-kg
/collections/all/products/apple-shimla-price-per-500gms
/collections/all/products/apple-red-delicious-imported-price-per-500-gms
/collections/all/products/apple-royal-gala-price-per-500-gms
/collections/all/products/apple-washington
/collections/all/products/apricot-dry-fresh-price-per-200gms
/collections/all/products/arbi-price-per-250-gms
/collections/all/products/asparagus-thai
/collections/all/products/avacado-hass-imported-price-per-pc
/collections/all/products/avacados
/collections/all/products/baby-corn-price-per-200gms
/collections/all/products/banana
/collections/all/products/banana-flower-price-per-pcs-approx-400gms-to-500gms
/collections/all/products/banana-raw-price-per-500-gms


https://fastandfresh.in/collections/all?page=2
/collections/all/products/basil-leaves
/collections/all/products/basket-ff

ConnectionError: HTTPSConnectionPool(host='fastandfresh.in', port=443): Max retries exceeded with url: /collections/all/products/star-fruit-kamrakh-price-per-250gms (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f9850ae20d0>: Failed to establish a new connection: [Errno 60] Operation timed out'))

# Look at data setup and save

In [19]:
df = pd.DataFrame(data_list)

In [20]:
df.shape

(89, 14)

In [21]:
df.columns

Index(['@context', '@type', 'offers', 'brand', 'name', 'description',
       'category', 'url', 'sku', 'image', 'availability', 'price',
       'priceCurrency', 'priceValidUntil'],
      dtype='object')

In [22]:
dt = str(date.today())
dt

'2021-02-09'

In [23]:
df['date'] = dt

In [24]:
df.to_csv('scraped_data/FastnFresh_'+dt+'.csv',index=0)